# 03 — Feature Engineering: X Closest Users

**Project #8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**  
Network Data Analysis Laboratory 2025-2026, Politecnico di Milano

---

## Team-8 Specific Feature

For each user sample, augment the feature vector with the features of the **X spatially closest users** at the same timestamp, using 3D Euclidean distance on `(x, y, z)` coordinates.

**Features copied from each neighbour:**
`sinr_dl, sinr_ul, prb, bler, traffic_type, throughput`

**Values of X to experiment with:** 1, 3, 5, 10

This notebook:
1. Builds the enriched DataFrame for each X value
2. Runs full preprocessing (scale + split)
3. Saves the feature-enriched processed arrays

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

from src.data.loader import load_venue
from src.data.preprocessor import Preprocessor
from src.features.closest_users import (
    add_closest_user_features,
    get_neighbor_column_names,
    NEIGHBOR_FEATURES,
)

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
X_VALUES = [1, 3, 5, 10]  # <-- experiment with different X

## 1. Load raw ACC Arena data

In [ ]:
acc = load_venue('acc_arena', data_dir='../data/raw')
print(f"Raw shape: {acc.shape}")
acc.head(3)

## 2. Build closest-user features for each X

In [ ]:
# NOTE: This step can be slow for large datasets.
# Consider running with a subset first (e.g., nrows=100_000) to validate,
# then re-run on the full dataset.

enriched = {}   # {X_value: enriched_DataFrame}

for x_val in X_VALUES:
    print(f"\nBuilding features for X={x_val} …")
    df_x = add_closest_user_features(acc, X=x_val)
    enriched[x_val] = df_x
    neighbor_cols = get_neighbor_column_names(x_val)
    print(f"  Added {len(neighbor_cols)} columns. Total shape: {df_x.shape}")

In [ ]:
# Inspect added columns for X=3
x_demo = 3
neighbor_cols_demo = get_neighbor_column_names(x_demo)
print(f"New columns for X={x_demo}:")
print(neighbor_cols_demo)
display(enriched[x_demo][neighbor_cols_demo].describe())

## 3. Visualise neighbour feature distributions

In [ ]:
# Compare own SINR vs. nearest-neighbour SINR (X=5)
x_val = 5
df5 = enriched[x_val]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat, col_name in [
    (axes[0], 'sinr_dl',      'Own DL SINR'),
    (axes[1], 'neighbor_1_sinr_dl', 'Nearest-Neighbor DL SINR'),
]:
    ax.hist(df5[feat].dropna(), bins=60, color='steelblue', alpha=0.7)
    ax.set_xlabel('SINR (dB)')
    ax.set_title(col_name)

plt.suptitle(f'SINR: own vs. nearest neighbour (X={x_val})')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of neighbour throughput vs. own throughput
fig, axes = plt.subplots(1, len(X_VALUES), figsize=(5 * len(X_VALUES), 4))

for ax, x_val in zip(axes, X_VALUES):
    df = enriched[x_val]
    nb_throughputs = [f'neighbor_{r}_throughput' for r in range(1, x_val + 1)]
    avg_nb_tp = df[nb_throughputs].mean(axis=1)
    ax.scatter(avg_nb_tp.sample(3000, random_state=42),
               df.loc[avg_nb_tp.sample(3000, random_state=42).index, 'throughput'],
               s=5, alpha=0.3, color='steelblue')
    ax.set_xlabel('Avg neighbour throughput (Mbps)')
    ax.set_ylabel('Own throughput (Mbps)')
    ax.set_title(f'X={x_val}')

plt.suptitle('Own vs. Average Neighbour Throughput')
plt.tight_layout()
plt.show()

## 4. Preprocessing with neighbour features

In [ ]:
preprocessed = {}  # {X_value: (X_train, X_val, X_test, y_train, y_val, y_test)}
preprocessors = {}

for x_val in X_VALUES:
    neighbor_cols = get_neighbor_column_names(x_val)
    prep = Preprocessor(random_state=RANDOM_SEED)
    splits = prep.fit_transform(enriched[x_val], extra_feature_cols=neighbor_cols)
    preprocessed[x_val]  = splits
    preprocessors[x_val] = prep
    print(f"X={x_val:2d} — train shape: {splits[0].shape}")

## 5. Save enriched feature arrays

In [ ]:
for x_val in X_VALUES:
    X_train, X_val_, X_test, y_train, y_val_, y_test = preprocessed[x_val]
    prefix = PROCESSED_DIR / f'acc_X{x_val}'
    np.save(f'{prefix}_X_train.npy', X_train)
    np.save(f'{prefix}_X_val.npy',   X_val_)
    np.save(f'{prefix}_X_test.npy',  X_test)
    np.save(f'{prefix}_y_train.npy', y_train)
    np.save(f'{prefix}_y_val.npy',   y_val_)
    np.save(f'{prefix}_y_test.npy',  y_test)
    preprocessors[x_val].save(f'{prefix}_preprocessor.pkl')
    print(f"Saved X={x_val} arrays.")

print('\nAll feature-enriched arrays saved.')

## 6. Feature dimension summary

| X  | Base features | Added neighbour features | Total |
|----|--------------|--------------------------|-------|
| 0  | …            | 0                        | …     |
| 1  | …            | 6                        | …     |
| 3  | …            | 18                       | …     |
| 5  | …            | 30                       | …     |
| 10 | …            | 60                       | …     |